# Does vision resolve depth / width / height where OCR text cannot?

A zero-shot probe on 400 held-out rows.

**The problem.** OCR reads `9.3cm` off the product photo reliably — the answer is
present in the text **71.7%** of the time. But nothing in the text says *which
dimension* it is, so the rules-plus-tagger cascade stalls at **36.2%**. 97% of
these rows have no `A x B x C` triple and no labelled dimension, so there is
nothing textual left to exploit. The information lives in the image's arrows and
callouts.

**Why this is not capped at 71.7%.** That ceiling bounds *text* pipelines. A VLM
reads pixels, so it can in principle exceed it.

| Reference (full val length set) | dim |
|---|---|
| S0 rules over OCR | 31.8% |
| tagger + rules cascade | **36.2%** ← the bar |
| OCR recoverability ceiling | 71.7% (text only) |

---

### Before running

1. **Settings panel → Accelerator → `GPU T4 x2`** (or P100).
2. **Settings panel → Internet → On.** Needed to pull the model from Hugging
   Face. Kaggle requires a phone-verified account for this.
3. **Add Input → Datasets → Your Datasets**, and attach the dataset holding
   `vlm_probe_sample.tar.gz` (21 MB). Kaggle unpacks the archive on upload, so
   the notebook accepts either the extracted `vlm_probe/` folder or the raw
   tarball. Any dataset name works — the next cell locates it by glob.

Then run all cells. Takes about 5 minutes on a T4.

In [ ]:
import glob, os, shutil, subprocess, sys, tarfile

# Kaggle unpacks archives when a dataset is created, so the files usually arrive
# already extracted under /kaggle/input/<name>/vlm_probe/. Look for the extracted
# sample.csv first and read it in place (read-only is fine); fall back to
# extracting a tarball if one was attached instead.
found = (glob.glob("/kaggle/input/**/sample.csv", recursive=True)
         + glob.glob("**/sample.csv", recursive=True))
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

if found:
    DEST = os.path.dirname(os.path.abspath(found[0]))
    print("using extracted data:", DEST)
else:
    tars = (glob.glob("/kaggle/input/**/vlm_probe_sample.tar.gz", recursive=True)
            + glob.glob("**/vlm_probe_sample.tar.gz", recursive=True))
    if not tars:
        raise SystemExit(
            "Neither sample.csv nor vlm_probe_sample.tar.gz found under /kaggle/input.\n"
            "Attach the dataset via Add Input > Datasets > Your Datasets."
        )
    print("extracting:", tars[0])
    with tarfile.open(tars[0]) as t:
        t.extractall(WORK)
    DEST = os.path.join(WORK, "vlm_probe")

assert os.path.isfile(f"{DEST}/sample.csv"), f"sample.csv missing in {DEST}"
assert os.path.isdir(f"{DEST}/images"), f"images/ missing in {DEST}"
print(f"ready: {len(os.listdir(DEST + '/images'))} images")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    print("\n!! No GPU. Settings panel > Accelerator > GPU T4 x2, then re-run.")

In [ ]:
# Kaggle's preinstalled transformers is often too old for Qwen2-VL.
import transformers, packaging.version as V
need = V.parse("4.45")
if V.parse(transformers.__version__) < need:
    print(f"transformers {transformers.__version__} is too old; upgrading...")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U",
                    "transformers>=4.45", "accelerate"], check=True)
    print("\n*** Now do Run > Restart & run all — the new version needs a fresh kernel. ***")
else:
    print(f"transformers {transformers.__version__} is fine")

In [ ]:
import csv, json, re, time
from collections import Counter

import torch
from PIL import Image

sys.path.insert(0, DEST)
import normalize as N          # identical taxonomy to the local pipeline

MODEL = "Qwen/Qwen2-VL-2B-Instruct"
PHRASE = {"depth": "depth (front to back)",
          "width": "width (left to right)",
          "height": "height (top to bottom)"}
PROMPT = (
    "This is a product photograph, which may include printed dimension labels or arrows.\n"
    "What is the product's {phrase}?\n"
    'Answer with only a number and a unit, for example "12 cm" or "4.5 in".\n'
    'If the {attr} is not shown, answer exactly "unknown".'
)

_ANS = re.compile(r"(-?\d+(?:[.,]\d+)?)\s*([A-Za-z\"'.]+)")

def parse_answer(raw):
    """Free-form model text -> (value, canonical length unit) or None."""
    if not raw or "unknown" in raw.lower():
        return None
    m = _ANS.search(raw.replace("\u2033", "in").replace("\u201d", "in"))
    if not m:
        return None
    unit = N.normalize_unit(m.group(2).strip("\"'."))
    if unit is None or N.family(unit) != "length":
        return None
    return float(m.group(1).replace(",", ".")), unit

def dim_correct(pred, lo, hi, gold_unit, tol=0.01):
    """The local pipeline's `dim` metric: same family, same magnitude in base
    units, gold treated as an interval."""
    if pred is None:
        return False
    v, u = pred
    if N.family(u) != N.family(gold_unit):
        return False
    a = N.to_base(v, u)
    return N.to_base(lo, gold_unit) * (1 - tol) <= a <= N.to_base(hi, gold_unit) * (1 + tol)

print("helpers ready")

In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

dev = "cuda" if torch.cuda.is_available() else "cpu"
proc = AutoProcessor.from_pretrained(MODEL, min_pixels=256*28*28, max_pixels=768*768)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL, dtype=torch.float16).to(dev).eval()
print("model loaded on", dev)

In [ ]:
rows = list(csv.DictReader(open(f"{DEST}/sample.csv")))
print(f"probing {len(rows)} length rows\n")

hits, out = Counter(), []
t0 = time.time()
for i, r in enumerate(rows):
    img = Image.open(f"{DEST}/images/{r['image_file']}").convert("RGB")
    a = r["attribute"]
    msgs = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": PROMPT.format(phrase=PHRASE[a], attr=a)}]}]
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[text], images=[img], return_tensors="pt").to(dev)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=24, do_sample=False)
    raw = proc.batch_decode(gen[:, inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)[0].strip()

    pred = parse_answer(raw)
    ok = dim_correct(pred, float(r["gold_lo"]), float(r["gold_hi"]), r["gold_unit"])
    hits["n"] += 1
    hits["answered"] += pred is not None
    hits["dim"] += ok
    hits[f"{a}_n"] += 1
    hits[f"{a}_dim"] += ok
    out.append({**r, "vlm_raw": raw,
                "vlm_pred": list(pred) if pred else None, "dim": bool(ok)})
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(rows)}  {(time.time()-t0)/(i+1):.2f}s/img  "
              f"dim {100*hits['dim']/hits['n']:.1f}%", flush=True)

RESULTS = f"{WORK}/vlm_probe_results.json"
json.dump(out, open(RESULTS, "w"), indent=1)
print("\nwrote", RESULTS)

In [ ]:
n = hits["n"]
print(f"probed {n} rows in {(time.time()-t0)/60:.1f} min\n")
print(f"{'system':<30}{'coverage':>10}{'dim':>8}")
print("-" * 48)
print(f"{'VLM zero-shot (Qwen2-VL-2B)':<30}{100*hits['answered']/n:>9.1f}%{100*hits['dim']/n:>7.1f}%")
print(f"{'tagger + rules cascade':<30}{'':>10}{36.2:>7.1f}%")
print(f"{'S0 rules over OCR':<30}{'':>10}{31.8:>7.1f}%")
print(f"{'OCR ceiling (text only)':<30}{'':>10}{71.7:>7.1f}%")

print(f"\n{'attribute':<12}{'n':>5}{'VLM dim':>10}")
print("-" * 28)
for a in ("depth", "width", "height"):
    k = hits[f"{a}_n"] or 1
    print(f"{a:<12}{hits[f'{a}_n']:>5}{100*hits[f'{a}_dim']/k:>9.1f}%")

# A 400-row sample carries a real interval; quote it, don't quote a point estimate.
import math
p = hits["dim"] / n
z = 1.96
d = 1 + z*z/n
c = p + z*z/(2*n)
h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
print(f"\n95% CI on the VLM figure: [{100*(c-h)/d:.1f}, {100*(c+h)/d:.1f}]")

print("""
Download vlm_probe_results.json (right panel > Output > Download) and hand it
back — it holds the raw model text per row, so the failures can be read
directly rather than inferred from an aggregate.
""")